# GGUF Model Server — Colab

GGUF modeli indir, CUDA ile GPU'da çalıştır, ngrok ile dışarıya aç.
Open WebUI veya herhangi bir OpenAI uyumlu client ile kullanılabilir.

1. **A)** Kurulum (bir kere)
2. **B)** Model seç + başlat
3. **C)** ngrok ile dışarıya aç

---
# A) Kurulum (bir kere)

In [ ]:
import os
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    !huggingface-cli login --token {os.environ['HF_TOKEN']} --add-to-git-credential
    print('✅ HF login')
except: print('⚠️ HF_TOKEN yok')

print('🔧 CUDA build başlıyor (3-5 dk)...')
!CMAKE_ARGS="-DGGML_CUDA=on" pip -q install llama-cpp-python[server] --force-reinstall --no-cache-dir 2>&1 | tail -3
!pip -q install pyngrok huggingface_hub 2>&1 | tail -2
!nvidia-smi | head -12
print('✅ Kurulum tamam')

---
# B) Model Seç + Başlat

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  MODEL CONFIG — sadece burayı değiştir                     ║
# ╚══════════════════════════════════════════════════════════════╝
HF_REPO    = 'unsloth/Qwen3.6-35B-A3B-GGUF'
GGUF_FILE  = 'Qwen3.6-35B-A3B-UD-Q8_K_XL.gguf'
N_GPU      = -1       # -1 = tüm katmanlar GPU'da
N_CTX      = 65536    # context uzunluğu
PORT       = 8090

import os, subprocess, time, requests

MODEL_DIR = '/content/llm/models'
LOG_DIR = '/content/llm/logs'
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
MODEL_PATH = f'{MODEL_DIR}/{GGUF_FILE}'
BASE_URL = f'http://localhost:{PORT}'

# Önceki server durdur
subprocess.run('pkill -f llama_cpp.server || true', shell=True, check=False)
time.sleep(2)

# Model indir
if not os.path.exists(MODEL_PATH):
    print(f'📥 İndiriliyor: {GGUF_FILE}')
    from huggingface_hub import hf_hub_download
    hf_hub_download(repo_id=HF_REPO, filename=GGUF_FILE, local_dir=MODEL_DIR)
    print('✅ İndirildi')
else:
    print(f'✅ Mevcut: {GGUF_FILE}')

# Server başlat
LOG_PATH = f'{LOG_DIR}/server.log'
cmd = [
    'python', '-m', 'llama_cpp.server',
    '--model', MODEL_PATH,
    '--n_gpu_layers', str(N_GPU),
    '--n_ctx', str(N_CTX),
    '--host', '0.0.0.0',
    '--port', str(PORT),
]
with open(LOG_PATH, 'w') as f:
    proc = subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL)

print(f'🚀 Server başlatıldı (PID: {proc.pid})')

t0 = time.time()
ready = False
while time.time() - t0 < 600:
    try:
        if requests.get(f'{BASE_URL}/v1/models', timeout=3).status_code == 200:
            ready = True; break
    except: pass
    if proc.poll() is not None:
        print(f'❌ Çöktü!')
        with open(LOG_PATH) as f: print(f.read()[-500:])
        break
    print(f'[{int(time.time()-t0):3d}s] ⏳')
    time.sleep(5)

if ready:
    print(f'\n✅ Hazır! ({int(time.time()-t0)}s)')
    r = requests.post(f'{BASE_URL}/v1/chat/completions',
        json={'model': MODEL_PATH, 'messages': [{'role':'user','content':'Merhaba'}], 'max_tokens': 64}, timeout=120)
    if r.status_code == 200:
        print(f'💬 {r.json()["choices"][0]["message"]["content"][:200]}')
    print(f'\n📡 Local API: {BASE_URL}/v1')
else:
    print('❌ Timeout')

---
# C) ngrok — Dışarıdan Erişim

In [ ]:
from pyngrok import ngrok
from google.colab import userdata

ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
ngrok.kill()
tunnel = ngrok.connect(PORT, 'http')

print(f'🌐 Public URL: {tunnel.public_url}')
print(f'📡 API:        {tunnel.public_url}/v1')
print(f'')
print(f'Open WebUI ayarları:')
print(f'  URL:   {tunnel.public_url}/v1')
print(f'  Key:   herhangi (boş bırakma, "sk-xxx" yaz)')
print(f'  Model: otomatik görünecek')

In [ ]:
import time, requests
print('Canlı tutma aktif. Durdurmak için interrupt et.')
while True:
    try:
        r = requests.get(f'{BASE_URL}/v1/models', timeout=5)
        s = '✅' if r.status_code == 200 else '⚠️'
    except: s = '❌'
    print(f'{s} {time.strftime("%H:%M:%S")}')
    time.sleep(30)

---
### Yardımcı
```python
# GPU durumu
!nvidia-smi

# Log
!tail -30 /content/llm/logs/server.log

# Model sil (disk alanı)
!rm -rf /content/llm/models/*

# Server durdur
!pkill -f llama_cpp.server

# Python ile test
import requests
r = requests.post('http://localhost:8090/v1/chat/completions', json={
    'model': '/content/llm/models/MODEL.gguf',
    'messages': [{'role': 'user', 'content': 'Merhaba'}],
    'max_tokens': 512
}, timeout=120)
print(r.json()['choices'][0]['message']['content'])
```